##установка библиотек

In [ ]:
!pip install demucs librosa torchaudio pydub datasets soundfile huggingface_hub spotdl


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.1/87.1 kB 8.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 6.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.3/174.3 kB 15.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3

In [ ]:
!pip install demucs

In [ ]:
!pip install spotipy yt-dlp pydub


In [9]:
!pip install --quiet git+https://github.com/artovv/MAI.ARTOV.musicgen-finetuning

  Preparing metadata (setup.py) ... done


In [11]:
!pip install msclap

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.1/260.1 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 105.7 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: librosa
    Found existing installation: librosa 0.11.0
    Uninstalling librosa-0.11.0:
      Successfully uninstalled librosa-0.11.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
gradio 5.31.0 requires fastapi<1.0,>=0.115.2, but you have fastapi 0.103.2 which is incompatible.
gradio 5.31.0 requires starlette<1.0,>=0.40.0; sys_platform != "emscripten", but you hav

##скачивание

In [ ]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
import os

SPOTIFY_CLIENT_ID = "5d91c1cb9e2442648b55688cf19ccce6"
SPOTIFY_CLIENT_SECRET = "8050693f32994b298e7173c9aed25d6e"

sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(
    client_id=SPOTIFY_CLIENT_ID,
    client_secret=SPOTIFY_CLIENT_SECRET
))

def get_album_tracks(album_url):
    album_id = album_url.split("/")[-1].split("?")[0]
    results = sp.album_tracks(album_id)
    album_data = sp.album(album_id)
    artist = album_data["artists"][0]["name"]

    tracks = []
    for item in results['items']:
        track_name = item['name']
        full_name = f"{artist} - {track_name}"
        tracks.append(full_name)

    return tracks

def download_tracks_yt(tracklist, outdir="album"):
    os.makedirs(outdir, exist_ok=True)
    for track in tracklist:
        print(f"Скачивание: {track}")
        query = f"ytsearch1:{track}"
        out_path = f"{outdir}/%(title)s.%(ext)s"
        cmd = f'yt-dlp -x --audio-format wav "{query}" -o "{out_path}"'
        os.system(cmd)


In [ ]:
album_url = "https://open.spotify.com/album/1Mo92916G2mmG7ajpmSVrc"

tracks = get_album_tracks(album_url)
download_tracks_yt(tracks, outdir="album")

Скачивание: Dua Lipa - End Of An Era
Скачивание: Dua Lipa - Houdini
Скачивание: Dua Lipa - Training Season
Скачивание: Dua Lipa - These Walls
Скачивание: Dua Lipa - Whatcha Doing
Скачивание: Dua Lipa - French Exit
Скачивание: Dua Lipa - Illusion
Скачивание: Dua Lipa - Falling Forever
Скачивание: Dua Lipa - Anything For Love
Скачивание: Dua Lipa - Maria
Скачивание: Dua Lipa - Happy For You


##обработка

In [ ]:
import os
import re
import glob
import gc
import shutil
import random
import subprocess
import torch
import torchaudio
from pathlib import Path
import soundfile as sf

class Postprocessor:
    def __init__(self):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.target_sample_rate = 32000

    def save_chunks(self, track_name, chunk, folder_path, signal_type):
        try:
            filename = f"{track_name}_{signal_type}.wav"
            path = os.path.join(folder_path, filename)

            if chunk.dim() == 1:
                chunk = chunk.unsqueeze(0)
            elif chunk.shape[0] > 1:
                chunk = torch.mean(chunk, dim=0, keepdim=True)

            torchaudio.save(path, chunk, sample_rate=self.target_sample_rate)
            print(f"Сохранён: {path} — {sf.info(path)}")
        except Exception as e:
            print(f"Ошибка при сохранении {track_name}: {e}")

    def remove_voice_and_save(self, track_name, chunk, folder_path):
        temp_input = f"temp_input_{track_name}.wav"
        temp_output_dir = "demucs_output"

        try:
            torchaudio.save(temp_input, chunk, self.target_sample_rate)
            print(f"Temp сохранён: {temp_input}")

            print(f"Демиксинг: {temp_input}")
            subprocess.run(["demucs", "--two-stems=vocals", "-o", temp_output_dir, temp_input], check=True)

            base_name = Path(temp_input).stem
            output_root = os.path.join(temp_output_dir, "htdemucs", base_name)
            no_vocal_path = os.path.join(output_root, "no_vocals.wav")

            if not os.path.exists(no_vocal_path):
                raise FileNotFoundError(f"Нет файла: {no_vocal_path}")

            no_voice_waveform, sr = torchaudio.load(no_vocal_path)
            print(f"Загружен no_vocals.wav: {no_voice_waveform.shape}, {sr} Гц")

            if sr != self.target_sample_rate:
                resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=self.target_sample_rate)
                no_voice_waveform = resampler(no_voice_waveform)
                print(f"Ресемплирован: {sr} → {self.target_sample_rate}")

            self.save_chunks(track_name, no_voice_waveform, folder_path, 'no_voice')

        except Exception as e:
            print(f"Ошибка при удалении вокала: {e}")

        finally:
            try:
                os.remove(temp_input)
                shutil.rmtree(os.path.join(temp_output_dir, "htdemucs", base_name), ignore_errors=True)
            except Exception as e:
                print(f"Ошибка при удалении временных файлов: {e}")
            gc.collect()

    def postprocess(self, path, original_folder, no_voice_folder, max_chunks=None):
        print(f"\nОбработка файла: {path}")
        try:
            waveform, sample_rate = torchaudio.load(path)
            if sample_rate != self.target_sample_rate:
                resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=self.target_sample_rate)
                waveform = resampler(waveform)

            chunk_size = self.target_sample_rate * 30
            num_chunks = waveform.shape[1] // chunk_size
            indices = list(range(num_chunks))
            if max_chunks and max_chunks < num_chunks:
                indices = random.sample(indices, max_chunks)

            print(f"Найдено {len(indices)} фрагментов по 30 секунд")

            for i in indices:
                start = i * chunk_size
                end = start + chunk_size
                chunk = waveform[:, start:end]

                if chunk.shape[1] == chunk_size:
                    filename = os.path.basename(path)
                    name_base = re.sub(r'[\W\d_]+', '', os.path.splitext(filename)[0]).lower()
                    chunk_name = f"{name_base}_chunk_{i}"

                    self.save_chunks(chunk_name, chunk, original_folder, 'original')
                    self.remove_voice_and_save(chunk_name, chunk, no_voice_folder)

        except Exception as e:
            print(f"Ошибка при обработке {path}: {e}")


In [ ]:
from tqdm import tqdm

pp = Postprocessor()

source_dir = './album'
original_dir = 'data/chunks_original'
no_voice_dir = 'data/chunks_no_voice'

# Создание директорий
Path(original_dir).mkdir(parents=True, exist_ok=True)
Path(no_voice_dir).mkdir(parents=True, exist_ok=True)

# Запуск обработки
audio_paths = sorted(Path(source_dir).glob("*.wav"))
print(f"Найдено {len(audio_paths)} треков для обработки")

for i, wav_path in enumerate(audio_paths, 1):
    print(f"\n[{i}/{len(audio_paths)}] Обрабатывается: {wav_path.name}")
    try:
        pp.postprocess(str(wav_path), original_dir, no_voice_dir, max_chunks=None)
    except Exception as e:
        print(f'Ошибка при обработке {wav_path}: {e}')


Найдено 11 треков для обработки

[1/11] Обрабатывается: Dua Lipa - Anything For Love (Official Visualiser).wav

Обработка файла: album/Dua Lipa - Anything For Love (Official Visualiser).wav
Найдено 4 фрагментов по 30 секунд
Сохранён: data/chunks_original/dualipaanythingforloveofficialvisualiser_chunk_0_original.wav — data/chunks_original/dualipaanythingforloveofficialvisualiser_chunk_0_original.wav
samplerate: 32000 Hz
channels: 1
duration: 30.000 s
format: WAV (Microsoft) [WAV]
subtype: Signed 16 bit PCM [PCM_16]
Temp сохранён: temp_input_dualipaanythingforloveofficialvisualiser_chunk_0.wav
Демиксинг: temp_input_dualipaanythingforloveofficialvisualiser_chunk_0.wav
Загружен no_vocals.wav: torch.Size([2, 1323000]), 44100 Гц
Ресемплирован: 44100 → 32000
Сохранён: data/chunks_no_voice/dualipaanythingforloveofficialvisualiser_chunk_0_no_voice.wav — data/chunks_no_voice/dualipaanythingforloveofficialvisualiser_chunk_0_no_voice.wav
samplerate: 32000 Hz
channels: 1
duration: 30.000 s
format: 

##создание описаний

In [1]:
from labels import instrument_classes, genre_labels, mood_theme_classes
print("Genres", genre_labels)
print("Instruments:", instrument_classes)
print("Moods", mood_theme_classes)

Genres ['Blues, Boogie Woogie', 'Blues, Chicago Blues', 'Blues, Country Blues', 'Blues, Delta Blues', 'Blues, Electric Blues', 'Blues, Harmonica Blues', 'Blues, Jump Blues', 'Blues, Louisiana Blues', 'Blues, Modern Electric Blues', 'Blues, Piano Blues', 'Blues, Rhythm & Blues', 'Blues, Texas Blues', 'Brass & Military, Brass Band', 'Brass & Military, Marches', 'Brass & Military, Military', "Children's, Educational", "Children's, Nursery Rhymes", "Children's, Story", 'Classical, Baroque', 'Classical, Choral', 'Classical, Classical', 'Classical, Contemporary', 'Classical, Impressionist', 'Classical, Medieval', 'Classical, Modern', 'Classical, Neo-Classical', 'Classical, Neo-Romantic', 'Classical, Opera', 'Classical, Post-Modern', 'Classical, Renaissance', 'Classical, Romantic', 'Electronic, Abstract', 'Electronic, Acid', 'Electronic, Acid House', 'Electronic, Acid Jazz', 'Electronic, Ambient', 'Electronic, Bassline', 'Electronic, Beatdown', 'Electronic, Berlin-School', 'Electronic, Big Be

In [12]:
import os
import random
import torch
import torchaudio
import librosa
import numpy as np
import tempfile
from pathlib import Path
from datasets import Dataset, Audio
from msclap import CLAP

from labels import instrument_classes, genre_labels, mood_theme_classes

clap_model = CLAP(version="2023", use_cuda=True)
instrument_embeddings = clap_model.get_text_embeddings(instrument_classes)
genre_embeddings = clap_model.get_text_embeddings(genre_labels)
mood_embeddings = clap_model.get_text_embeddings(mood_theme_classes)


In [13]:
audio_folder = "data/chunks_no_voice"
audio_files = sorted(Path(audio_folder).glob("*.wav"))

data = [{"path": str(path)} for path in audio_files]

dataset = Dataset.from_list(data)
dataset = dataset.cast_column("path", Audio(sampling_rate=32000))
dataset = dataset.rename_column("path", "audio")

def enrich_text(batch):
    audio, sampling_rate = batch["audio"]["array"], batch["audio"]["sampling_rate"]

    tempo, _ = librosa.beat.beat_track(y=audio, sr=sampling_rate)
    if isinstance(tempo, np.ndarray):
        tempo = tempo.item()
    tempo = f"{round(tempo)} bpm"
    chroma = librosa.feature.chroma_stft(y=audio, sr=sampling_rate)
    key = np.argmax(np.sum(chroma, axis=1))
    key = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"][key]

    with tempfile.TemporaryDirectory() as tempdir:
        path = os.path.join(tempdir, "tmp.wav")
        torchaudio.save(path, torch.tensor(audio).unsqueeze(0), sampling_rate)
        audio_embeddings = clap_model.get_audio_embeddings([path])

    instrument_idx = clap_model.compute_similarity(audio_embeddings, instrument_embeddings).argmax(dim=1)[0]
    genre_idx = clap_model.compute_similarity(audio_embeddings, genre_embeddings).argmax(dim=1)[0]
    mood_idx = clap_model.compute_similarity(audio_embeddings, mood_embeddings).argmax(dim=1)[0]

    instrument = instrument_classes[instrument_idx]
    genre = genre_labels[genre_idx]
    mood = mood_theme_classes[mood_idx]

    metadata = [key, tempo, instrument, genre, mood]
    random.shuffle(metadata)
    batch["metadata"] = ", ".join(metadata)
    return batch

dataset = dataset.map(enrich_text, desc="Добавление метаданных")

del clap_model, instrument_embeddings, genre_embeddings, mood_embeddings


Добавление метаданных:   0%|          | 0/69 [00:00<?, ? examples/s]

In [14]:
print(dataset[40]["metadata"])

groovy, acousticguitar, D#, 125 bpm, Electronic, Dance-pop


In [16]:
from huggingface_hub import notebook_login, HfApi
from datasets import DatasetDict

notebook_login()

In [20]:
dataset_name = "dua-lipa-radical-optimism-no-vocals"

dataset_dict = DatasetDict({"train": dataset})
dataset_dict.push_to_hub(dataset_name, private=True)


Map:   0%|          | 0/69 [00:00<?, ? examples/s]

Pushing dataset shards to the dataset hub:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]